# 5 - Úvod do PyTorch

Tento text slúži ako krátka **učebnicová kapitola**: nevychádza z jedného konkrétneho zadania, ale zostavuje **opakovateľnú štruktúru programu**, ktorú budete pri ďalších úlohách (spracovanie tabuliek, obrázkov, jednoduché sieťové modely) vedieť **prepísať len v miestach, kde sa mení samotná úloha** — nie od nuly a nie náhodným štýlom z internetu.

PyTorch je knižnica, v ktorej popisujete **výpočet** (model), **dáta** (ako ich načítavate a v akých dávkach ich sieť vidí) a **učenie** (ako z chyby počítate gradienty a ako meníte váhy). Nižšie prejdeme tieto vrstvy v poradí, v akom sa zvyknú skladať v reálnych skriptoch. Konkrétne súbory z hodiny, ktoré na tejto kostre stavajú, sú uvedené **až v závere** — najprv je dôležité pochopiť spoločný rámec.


## 1. Od dát po logovanie

V praxi sa osvedčuje držať sa **stále rovnakého sledu krokov**, lebo tak viete nájsť chyby (zlá normalizácia, zabudnutý `eval`, zlé poradie `zero_grad`).

Najprv nastavíte **reprodukovateľnosť** (`seed`) a zistíte, či počítate na **CPU alebo GPU** (`device`). Potom načítate dáta z disku alebo z ineho zdroju a rozdelíte ich na časť na učenie, časť na kontrolu počas učenia (validácia) a často aj na samostatný test, ktorý použijete až nakoniec. Pri **normalizácii** (odčítanie priemeru, delenie smerodajnou odchýlkou a podobne) musíte vždy odvodiť čísla **len z tréningovej časti** a tie isté transformácie potom aplikovať na validáciu a test — inak by ste do modelu prepašovali informáciu z budúcnosti.

Ďalej  zabalíte svoje dáta do triedy **`Dataset`** a z nej spravíte **`DataLoader`**, ktorý vám dáva dáta po **batchoch** a môže ich miešať. Definujete **model** ako podtriedu `nn.Module`, zvolíte **stratu** (`criterion`) a **optimalizátor**. Hlavná slučka beží po **epochách**: v každej najprv prejdete tréningové batchy (váhy sa menia), potom voliteľne validačné (váhy sa nemenia) a zapisujete si priebeh do logu alebo do **TensorBoardu**, aby ste videli, či sa učenie zlepšuje a či netrpíte pretreningom na tréningovej množine.


## 2. Tenzory a zariadenie (`device`)

Základná dátová štruktúra v PyTorch je **tenzor**. Je podobný poľu v NumPy, ale vie byť uložený na **GPU** a v režime učenia vie byť naviazaný na **automatické derivovanie** (gradienty). Pre vstupy a váhy sa bežne používa typ **`float32`**; dvojitá presnosť (`float64`) sa v neurónových sieťach často nevyžaduje a zbytočne zväčšuje pamäť.

**Zariadenie** (`cpu` alebo `cuda`) určuje, kde sa konkrétne čísla fyzicky nachádzajú. Pred operáciou medzi dvoma tenzormi musia byť **na tom istom** zariadení. Preto sa model raz presunie pomocou `.to(device)` a každý batch z `DataLoadera` tiež. Ak by jedna časť výpočtu ostala na CPU a druhá na GPU, dostanete chybu — je to zámerná ochrana pred tichým zmiešaním.

Gradienty sa zvyčajne nepočítajú pre samotné vstupné dáta, ale pre **parametre modelu** (váhy). To, či má tenzor `requires_grad`, určuje, či sa do neho má počítať spätný prechod; pri vstupných bodoch z datasetu to často nepotrebujete.


In [ ]:
import torch

# tensory z čísel
a = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
b = torch.ones(2, 3)

# zariadenie
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
a = a.to(device)

print("a:", a, "shape:", a.shape, "device:", a.device)
print("b shape:", b.shape)


### Rozbor predchádzajúcej bunky (tenzory a zariadenie)

Riadok `import torch` načíta hlavný modul. **`torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)`** vytvorí jednorozmerný tenzor z Python zoznamu; čísla s desatinnou bodkou sú dôležité — PyTorch z nich urobí **plávajúcu presnosť** vhodnú pre neurónové siete. Parameter **`dtype=torch.float32`** hovorí explicitne, že nechceme predvolené `float64`, ktoré by zdvojnásobilo pamäť bez prínosu pri základnom učení.

**`torch.ones(2, 3)`** vytvorí maticu samých jednotiek s tvarom **(2, 3)** — dva riadky, tri stĺpce. Atribút **`.shape`** (nižšie vypísaný pre `b`) je v PyTorch rovnaký pojem ako „rozmer“ poľa: koľko indexov potrebujete na jedno číslo (tu dva: riadok a stĺpec).

**`torch.device("cuda" if torch.cuda.is_available() else "cpu")`** vyberie reťazec `"cuda"` ak je nainštalovaný ovládač GPU a PyTorch ho vidí, inak `"cpu"`. Premenná `device` sa potom používa konzistentne pre všetky výpočty na jednom mieste.

**`a = a.to(device)`** **presunie** obsah tenzora `a` na vybrané zariadenie (ak už tam je, môže vrátiť `a` bezo zmeny). Tenzor `b` sme na zariadenie nepresunuli — v tejto ukážke sa s `b` ďalej nepočíta; keby ste chceli napríklad sčítať `a + b`, museli by byť **obidva** na tom istom `device`, inak PyTorch hlási chybu. To je zámer: zabráni sa tichému zmiešaniu CPU a GPU.

Výpis **`print("a:", a, "shape:", a.shape, "device:", a.device)`** ukáže hodnoty, tvar a kde tenzor leží — pri ladení je užitočné kontrolovať najmä **`device`**.


## 3. `Dataset` a `DataLoader`: prečo vôbec existujú

Úlohou objektu **`Dataset`** je odpovedať na otázku: „Čo je i-ta vzorka v mojej množine?“ Implementujete metódu `__len__` (koľko vzoriek máte) a `__getitem__` (vrátiť pár vstup–cieľ, zvyčajne ako tenzory). Nemusíte mať všetko naraz v pamäti; môžete v `__getitem__` načítať súbor z disku, orezať obrázok atď.

Objekt **`DataLoader`** obalí váš `Dataset` a pri iterácii vám vracia **batchy**: namiesto jedného vektora dostanete tenzor s prvým rozmerom rovným veľkosti batchu. Vie **miešať** poradie (`shuffle=True` na tréningu), aby sieť nevidela stále rovnaké poradie, a vie na pozadí pripravovať ďalší batch (`num_workers`; na niektorých systémoch sa kvôli stabilité ponechá 0).

Bez `DataLoadera` by ste museli ručne krájať pole indexov a skladať tenzory. Pri malých úlohách je síce niekedy možné pracovať s celým tréningom v jednom tenzore, no podľa bežných konvencií v PyTorchi sa odporúča používať rovnaký vzor ako pri väčších dátových úlohách.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class Toy2DDataset(Dataset):
    def __init__(self, n=100, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.x = torch.randn(n, 2, generator=g) * 0.8 + 0.3
        # jednoduchá pravidelnosť: horná polovica -> trieda 1
        self.y = (self.x[:, 1] > 0).long()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]

ds = Toy2DDataset(100)
loader = DataLoader(ds, batch_size=16, shuffle=True)

xb, yb = next(iter(loader))
print("batch x:", xb.shape, "batch y:", yb.shape)




### Rozbor `Dataset` a `DataLoader`

**`from torch.utils.data import Dataset, DataLoader`** — `Dataset` určuje, ako sa pristupuje k jednotlivým vzorkám dát, a `DataLoader` z neho pri iterácii vytvára batche.

Trieda **`Toy2DDataset`** dedí od **`Dataset`**. V metóde **`__init__`** sa vytvorí **`torch.Generator().manual_seed(seed)`**, teda generátor náhodných čísel s pevným seedom. Vďaka tomu pri opakovanom spustení vzniknú tie isté body, čo je dôležité pre reprodukovateľnosť výsledkov. Volanie **`torch.randn(n, 2, generator=g)`** vytvorí tenzor tvaru **`(n, 2)`**, teda `n` bodov v dvojrozmernom priestore. Násobenie a posun (`* 0.8 + 0.3`) len upravia ich rozptyl a polohu.

Riadok **`self.y = (self.x[:, 1] > 0).long()`** vytvára triedy podľa druhej súradnice bodu. Výraz **`self.x[:, 1]`** vyberie druhú súradnicu každého bodu a porovnanie s nulou vytvorí logický tenzor hodnôt `True` a `False`. Metóda **`.long()`** tieto hodnoty prevedie na celé čísla `1` a `0`, teda na formát, ktorý sa bežne používa napríklad pri **`CrossEntropyLoss`**.

Metóda **`__len__`** vracia počet vzoriek v datasete. `DataLoader` ju používa na určenie toho, koľko prvkov má k dispozícii. Metóda **`__getitem__(self, i)`** vracia i-tu dvojicu `(x, y)`, teda vstup a príslušný cieľ. V tomto prípade je `x` vektor dĺžky 2 a `y` je jedno celé číslo určujúce triedu.

Volanie **`DataLoader(ds, batch_size=16, shuffle=True)`** znamená, že sa vzorky z datasetu budú pri iterácii brať po skupinách veľkosti 16. Ak je nastavené **`shuffle=True`**, ich poradie sa pred každým prechodom premieša. `DataLoader` potom jednotlivé vzorky spojí do väčších tenzorov, takže namiesto jednej dvojice `(x, y)` dostanete celý batch. Výraz **`next(iter(loader))`** vezme prvý batch z iterácie cez `loader` a používa sa tu len na ukážku tvarov premenných **`xb`** a **`yb`**. Typicky má `xb` tvar **`(16, 2)`** a `yb` tvar **`(16,)`**, čo je presne formát, ktorý sa následne odovzdáva modelu a stratovej funkcii.

Používanie dvojice **`Dataset` + `DataLoader`** je v PyTorchi štandardný spôsob práce s dátami bez ohľadu na ich typ. Rovnaký vzor sa používa pri štruktúrovaných dátach, napríklad pri tabuľkách, aj pri neštruktúrovaných dátach, ako sú obrázky, text, zvuk alebo video. `Dataset` vždy definuje, čo presne znamená jedna vzorka a čo sa má vrátiť ako vstup a cieľ, zatiaľ čo `DataLoader` sa stará o batchovanie, miešanie poradia a iteráciu. Vďaka tomu ostáva tréningový kód rovnaký aj vtedy, keď sa mení typ dát; mení sa najmä spôsob načítania jednej vzorky v `__getitem__`.


### Ako sa líši `Dataset` podľa typu dát

Z pohľadu PyTorchu sa základný vzor nemení: vlastný dataset má spravidla implementovať `__len__` a `__getitem__`, zatiaľ čo `DataLoader` z neho robí iterovateľné batchy. Rozdiel je najmä v tom, čo predstavuje jedna vzorka a v akom tvare sa vracia vstup a cieľ. Tento vzor sa používa rovnako pri tabuľkových dátach aj pri obrazových úlohách.

Pri štruktúrovaných dátach býva jedna vzorka spravidla jeden riadok tabuľky. `__getitem__` preto typicky vyberie jeden riadok, prevedie vstupné stĺpce na číselný tenzor a vráti cieľ, napríklad triedu alebo reálnu hodnotu. Vstup má zvyčajne tvar jedného vektora vlastností, napríklad `(d,)`, kde `d` je počet príznakov.

```python
import torch
from torch.utils.data import Dataset
import pandas as pd

class TabularDataset(Dataset):
    def __init__(self, df, feature_cols, target_col):
        self.X = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.y = torch.tensor(df[target_col].values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]
```

---


Pri klasifikácii RGB obrázkov (jpg -> RGB -> 3 channels, png -> RGBA -> 4 channels) je jedna vzorka jeden obrázok a jedna trieda. `__getitem__` preto načíta súbor z disku, prevedie obrázok na tenzor a vráti dvojicu `(image, label)`. Vstup už nie je vektor vlastností, ale tenzor obrazu, typicky tvaru `(3, H, W)`, a cieľ je jedno celé číslo určujúce triedu. To zodpovedá bežnému používaniu datasetov v `torchvision`, kde obrazové datasety tiež implementujú `__getitem__` a `__len__`.

```python
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

class ImageClassificationDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, i):
        image = Image.open(self.image_paths[i]).convert("RGB")
        image = self.transform(image)
        label = torch.tensor(self.labels[i], dtype=torch.long)
        return image, label
```
---

Pri segmentácii je jedna vzorka stále jeden obrázok, ale cieľ už nie je jedno číslo. Namiesto jedného labelu pre celý obrázok sa používa **ground truth maska**, teda maska s rovnakými priestorovými rozmermi ako obrázok, v ktorej má každý pixel priradenú svoju triedu. `__getitem__` preto vracia dvojicu `(image, mask)`. To je hlavný rozdiel oproti klasifikácii. Pri klasifikácii je výstupom jeden label pre celý obrázok, kým pri segmentácii sa trieda určuje pre každý pixel osobitne.

<p align="center">
  <img src="https://lh5.googleusercontent.com/proxy/PI8_izhNtbPAHImQbhEKnZGsZrzSt0O9GE9W4qyILoSrezqqx9lZBe0QsJTUxseIKVnvr6qNuwMqdaDQHeDld49VYgrR104NUwg50fgO-xQVQX1Q7k0GMqeEp7dtDGrcXjBfPjqgNMymOl19D2UTpi2XExpa5mQTWRR7bHTiRQy-hrUHF_c" alt="Ukážka rozdielu medzi klasifikáciou, detekciou a segmentáciou">
</p>

Na obrázku je tento rozdiel dobre vidieť. Pri klasifikácii stačí povedať, čo sa na obrázku nachádza, napríklad „cat“. Pri segmentácii však nestačí určiť, že sa na obrázku nachádza mačka alebo pes. Treba určiť aj to, **ktoré konkrétne pixely** patria mačke, ktoré psovi a ktoré pozadiu. Práve to predstavuje ground truth maska. Farebné prekrytie vpravo je len vizualizácia tejto masky. V datasete sa maska zvyčajne ukladá ako celočíselný tenzor, kde každá hodnota zodpovedá jednej triede, napríklad `0 = pozadie`, `1 = mačka`, `2 = pes`. Pri tejto úlohe je tiež dôležité, aby sa pri prípadných transformáciách obraz a maska upravovali konzistentne. Segmentácia patrí medzi štandardné obrazové úlohy podporované v `torchvision`.

```python
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from torchvision import transforms

class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, image_transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.image_transform = image_transform or transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, i):
        image = Image.open(self.image_paths[i]).convert("RGB")
        mask = Image.open(self.mask_paths[i])

        image = self.image_transform(image)
        mask = torch.as_tensor(np.array(mask), dtype=torch.long)

        return image, mask
```

Zhrnutie je jednoduché. Pri tabuľkových dátach `Dataset` vracia jeden riadok vlastností a jeden cieľ. Pri klasifikácii obrázkov vracia jeden obraz a jednu triedu. Pri segmentácii vracia jeden obraz a celú masku tried po pixeloch. `DataLoader` sa potom používa rovnako vo všetkých troch prípadoch. Mení sa len obsah jednej vzorky, nie celkový princíp práce s dátami. ([docs.pytorch.org][1])





## 4. Model: trieda `nn.Module`

Model v PyTorchi sa zvyčajne zapisuje ako trieda, ktorá dedí z `nn.Module`. V metóde `__init__` sa definujú vrstvy modelu, najmä tie, ktoré obsahujú učené parametre, napríklad `nn.Linear`. V metóde `forward` sa potom určí, ako sa vstup spracuje na výstup. Vďaka dedeniu z `nn.Module` vie PyTorch evidovať parametre modelu, presúvať model medzi zariadeniami a prepínať ho medzi režimami `train()` a `eval()`.

Tvar výstupu poslednej vrstvy musí zodpovedať typu úlohy. Pri klasifikácii do viacerých tried model vracia pre každú triedu jednu výstupnú hodnotu. Ak je na výstupe len lineárna vrstva bez aktivačnej funkcie, ide len o číselné hodnoty, podľa ktorých model porovnáva jednotlivé triedy. Ak sa na výstupe použije vhodná aktivačná funkcia, napríklad softmax, tieto hodnoty sa dajú interpretovať ako pravdepodobnosti, teda ako miera istoty modelu, že vstup patrí do jednotlivých tried. Napríklad výstup `(0.8, 0.1, 0.1)` môžeme čítať tak, že model priraďuje prvej triede približne 80 %, druhej 10 % a tretej 10 %.

Pri regresii býva výstupom jedno alebo viac reálnych čísel. Rozdiel teda nie je v tom, že by PyTorch fungoval inak, ale v tom, čo presne má model predpovedať a s čím sa jeho výstup porovnáva.

Dá sa to predstaviť aj na jednoduchej analógii s ovocím. Predstavte si, že vstupom sú vlastnosti ovocia, napríklad farba, veľkosť, tvar a hmotnosť. Tieto vlastnosti tvoria vstupné rozmery modelu. Na základe nich sa model učí rozhodnúť, či ide o jablko, hrušku, pomaranč, melón alebo inú triedu. Každá vstupná hodnota teda predstavuje jednu vlastnosť a model sa z ich kombinácie (vzoru) snaží odhadnúť správnu triedu.

Najprv je užitočné pozrieť sa na matematický model neurónu, teda perceptrónu. Jeho výpočet sa dá zapísať takto:

$$
z = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b
$$

kde `x1` až `xn` sú vstupy, `w1` až `wn` sú váhy a `b` je bias.

Na výsledok sa potom aplikuje aktivačná funkcia:

$$
a = f(z) = f\left(\sum_{i=1}^{n} w_i x_i + b\right)
$$

kde `f` môže byť napríklad ReLU, tanh alebo iná aktivačná funkcia.

<p align="center">
  <img src="https://www.gabormelli.com/RKB/images/thumb/3/31/artificial-neuron-model.png/600px-artificial-neuron-model.png" alt="Štruktúra umelého neurónu">
</p>

Tento obrázok ukazuje základnú stavebnú jednotku siete, teda perceptrón. Vstupy $x_1$, $x_2$, $x_3$ až $x_n$ predstavujú vlastnosti jednej vzorky. Každý vstup má svoju váhu $w_1$, $w_2$, $w_3$ až $w_n$. Lineárna časť znamená, že sa každý vstup vynásobí svojou váhou a tieto príspevky sa potom sčítajú spolu s hodnotou `bias` -> b. Tým vznikne jedna výsledná hodnota, teda celkový vstup do neurónu.

Na túto výslednú hodnotu sa potom aplikuje aktivačná funkcia, ktorá ju ďalej upraví. Napríklad `ReLU` nechá kladné hodnoty bez zmeny a záporné nastaví na nulu. `Tanh` prevedie hodnotu do intervalu od `-1` do `1`. Dá sa teda povedať, že lineárna časť určuje, ako silno jednotlivé vstupy prispievajú do spoločného súčtu, a aktivačná funkcia rozhoduje, ako sa tento výsledok ďalej prenesie do ďalšej vrstvy.


Keď tomuto rozumieme pri jednom neuróne, ľahšie sa pozerá na celú vrstvu alebo celú sieť.

<p align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1400/1*FlJ_ZPMSlpCdeuAYzC0gPw.png" alt="Jednoduchá neurónová sieť">
</p>

Na tomto obrázku sú vľavo vstupy `x1` až `x4`. Môžete si ich predstaviť ako číselné vlastnosti jednej vzorky, napríklad pri ovocí by to mohla byť farba, veľkosť, hmotnosť a tvar. Každý neurón v ďalšej vrstve robí presne to isté, čo bolo vidieť na predchádzajúcom obrázku pri jednom neuróne. Zoberie všetky vstupy, vynásobí ich svojimi váhami, sčíta ich, pripočíta `bias` a výsledok pošle do aktivačnej funkcie.

Práve preto sa dá povedať, že `nn.Linear` je vrstva, ktorá robí lineárnu časť pre viac neurónov naraz. Na obrázku tomu zodpovedá stredná časť s maticou váh, vstupným vektorom a biasom. Krok označený ako `activation` potom predstavuje aktivačnú funkciu, napríklad `ReLU` alebo `Tanh`. Hodnoty $a_1$, $a_2$, $a_3$ vpravo sú už výstupy po tejto úprave.

Bez aktivačných funkcií by sa aj viac lineárnych vrstiev za sebou správalo len ako jedna väčšia lineárna vrstva, takže model by sa horšie učil zložitejšie vzťahy v dátach. Aktivačné funkcie sú preto dôležité najmä v skrytých vrstvách, kde modelu umožňujú zachytiť zložitejšie pravidlá.

In [ ]:
import torch.nn as nn

class TinyClassifier(nn.Module):
    def __init__(self, in_dim=2, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        return self.fc(x)  # výstup: logits pre každú triedu

model = TinyClassifier()
print(model)
print("počet parametrov:", sum(p.numel() for p in model.parameters()))


### Rozbor `TinyClassifier`

`import torch.nn as nn` importuje modul `torch.nn`, v ktorom sú definované vrstvy modelov, aktivačné funkcie aj stratové funkcie.

`class TinyClassifier(nn.Module)` znamená, že model dedí z `nn.Module`. Vďaka tomu PyTorch vie evidovať jeho parametre, presúvať model napríklad na GPU a správne prepínať medzi režimami `train()` a `eval()`.

`super().__init__()` zavolá konštruktor rodičovskej triedy `nn.Module`. Tento krok je potrebný, aby sa model správne inicializoval a aby sa jeho vrstvy a parametre korektne zaregistrovali.

`self.fc = nn.Linear(in_dim, num_classes)` vytvorí plne prepojenú lineárnu vrstvu. Táto vrstva vezme vstupný vektor dĺžky `in_dim` a vytvorí z neho výstup dĺžky `num_classes`. Vnútri obsahuje váhy a `bias`, teda učené parametre modelu.

Ak si to predstavíte na príklade ovocia, `in_dim` môže znamenať počet vlastností, napríklad farba, veľkosť, tvar a hmotnosť. `num_classes` potom znamená počet tried, napríklad jablko, hruška, pomaranč a melón. Model sa učí, ako z týchto vlastností vypočítať výstupné hodnoty pre jednotlivé triedy.

Didakticky sa to dá premostiť takto. Na obrázku s jedným neurónom je lineárna časť tvorená váhami `w1`, `w2`, `w3` a `bias`. Na obrázku s celou sieťou sa ten istý princíp opakuje vo viacerých neurónoch naraz. `nn.Linear` v PyTorchi teda nerobí nič magické. Len vo väčšom počte naraz vykoná to, čo robí jeden neurón na prvom obrázku.

`forward(self, x)` definuje dopredný priechod modelom. V tomto jednoduchom prípade sa vstup pošle len cez jednu lineárnu vrstvu. Výstupom sú hodnoty pre jednotlivé triedy. Väčšia hodnota znamená, že model danú triedu považuje za pravdepodobnejšiu než ostatné.

Ak by bola na výstupe ešte aktivačná funkcia, napríklad `softmax`, tieto hodnoty by sa dali interpretovať ako pravdepodobnosti. Vtedy by ste mohli povedať, s akou mierou istoty model predpokladá, že vstup patrí do jednotlivých tried.

Ak by ste chceli model rozšíriť o skrytú vrstvu a aktivačnú funkciu, mohol by vyzerať napríklad takto:

```python
import torch
import torch.nn as nn

class TinyClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x
```

V tomto zápise `fc1` zodpovedá prvej lineárnej vrstve, `act` zodpovedá aktivačnej funkcii a `fc2` vytvára konečný výstup modelu. Namiesto `nn.ReLU()` by ste mohli použiť aj `nn.Tanh()` alebo inú aktivačnú funkciu podľa typu úlohy.

`print(model)` vypíše štruktúru modelu, teda jeho vrstvy a podmoduly.

`sum(p.numel() for p in model.parameters())` spočíta celkový počet skalárnych parametrov modelu. Pri malých modeloch ide o malé číslo, pri väčších sieťach však počet parametrov rastie veľmi rýchlo.

Predchádzajúca bunka ukázala klasifikáciu do dvoch tried (dva výstupy logitov). Pri **regresii** by ste v poslednej vrstve mali **`Linear(..., 1)`** — jedno reálne číslo ako predikciu závislej premennej. Strata by už nebola krížová entropia, ale typicky **stredná štvorcová chyba** medzi predikciou a meraným `y`. Skryté vrstvy (`Linear` + nelinearita) môžu mať rovnaký tvar ako pri klasifikácii; líši sa **výstupný rozmer** a **matematický význam** výstupu (trieda vs. číslo). Takto je nastavená aj úloha v súbore 5b, ktorú rozoberá kapitola 10.


## 5. Strata (`criterion`) a optimalizátor

Pri učení modelu nestačí len vypočítať výstup. Potrebujeme aj určiť, **ako veľmi sa tento výstup líši od správnej odpovede**. Práve na to slúži **funkcia straty** označovaná ako `criterion`. Ako vstup dostane výstup modelu a správne cieľové hodnoty `y` a vráti jedno číslo, ktoré vyjadruje mieru chyby. Čím je toto číslo menšie, tým lepšie výstup modelu zodpovedá očakávanému výsledku.

Pri klasifikácii do viacerých tried sa často používa `CrossEntropyLoss`. Tá porovnáva výstup modelu so správnou triedou a penalizuje model vtedy, keď dáva vysoké skóre nesprávnym triedam alebo nízke skóre správnej triede. Pri regresii sa často používa `MSELoss` (Mean Square Error), ktorá meria, ako ďaleko je predpovedaná číselná hodnota od skutočnej hodnoty.

Keď už vieme zmerať chybu, potrebujeme model aj **upravovať tak, aby sa chyba zmenšovala**. Tu je dôležité rozlíšiť dve veci.

**Gradient descent** je všeobecný princíp učenia modelu. Myšlienka je jednoduchá: ak vieme zmerať chybu modelu, chceme meniť jeho parametre tak, aby sa táto chyba postupne zmenšovala. Gradient nám hovorí, ako citlivo chyba reaguje na zmenu jednotlivých parametrov a zároveň určuje smer, v ktorom chyba rastie najrýchlejšie. Ak sa teda chceme dostať k menšej chybe, musíme spraviť krok v opačnom smere.

**Optimalizátor** je konkrétny algoritmus, ktorý realizuje tento krok v praxi. **Gradient descent** je všeobecná myšlienka učenia modelu: chceme meniť parametre tak, aby sa chyba postupne zmenšovala. **Gradient** pritom hovorí, ktorým smerom treba parametre upraviť, aby chyba klesala. Optimalizátor potom túto informáciu skutočne využije a rozhodne, ako veľký krok sa spraví a akým spôsobom sa parametre zmenia.

Dá sa to predstaviť aj na jednoduchej analógii s autom. Model je ako **auto**, ktoré sa snaží dostať do cieľa. Funkcia straty je ako údaj z navigácie, ktorý hovorí, ako ďaleko ste od správnej trasy alebo od cieľa. Gradient je ako pokyn z navigácie, ktorým smerom treba otočiť volant, aby ste sa priblížili správnym smerom. **Gradient descent** je teda všeobecná stratégia, že sa chceme podľa týchto pokynov postupne približovať k cieľu. **Optimalizátor** je samotný vodič a spôsob riadenia auta.

Práve preto rôzne optimalizátory robia ten istý základný cieľ rôznym spôsobom. `SGD` je jednoduchý vodič, ktorý sa riadi len aktuálnym pokynom a vždy spraví priamy krok podľa momentálneho gradientu. `SGD with momentum` okrem aktuálneho pokynu berie do úvahy aj predchádzajúci pohyb, takže jazda býva plynulejšia a menej rozkmitaná. `Adam` ide ešte ďalej a prispôsobuje veľkosť krokov pre jednotlivé parametre zvlášť podľa priebehu učenia, takže pohyb býva často stabilnejší a efektívnejší.

Inými slovami, gradient descent hovorí **čo chceme robiť** - postupne znižovať chybu podľa gradientu - a optimalizátor určuje **ako presne to budeme robiť**. Cieľ zostáva rovnaký, mení sa len spôsob, akým sa k nemu model približuje.

Tento rozdiel je dobre vidieť aj na vizualizáciách optimizerov. Na prvom obrázku hviezda označuje minimum chyby a farebné čiary ukazujú, akou cestou sa k nemu jednotlivé optimalizátory približujú. Na druhom obrázku farba pozadia vyjadruje veľkosť chyby. Čím je modrá tmavšia, tým je chyba menšia a tým sme bližšie k lepšiemu riešeniu. Aj tam farebné čiary ukazujú trajektórie jednotlivých optimalizátorov.
<p align="center">
  <img src="https://www.ruder.io/content/images/2016/09/contours_evaluation_optimizers.gif" alt="Porovnanie trajektórií optimizerov na vrstevniciach chyby">
</p>

<p align="center">
  <img src="https://www.ruder.io/content/images/2016/09/saddle_point_evaluation_optimizers.gif" alt="Porovnanie trajektórií optimizerov v okolí saddle pointu">
</p>

Na prvom obrázku vidno, že jednotlivé optimalizátory sa síce snažia dostať k rovnakému minimu, ale každý z nich volí inú cestu. Na druhom obrázku je situácia náročnejšia, takže rozdiel medzi jednoduchšími a pokročilejšími optimalizátormi je ešte výraznejší.

Celý tréningový krok potom prebieha v tomto poradí. Najprv sa zavolá `optimizer.zero_grad()`, aby sa vymazali gradienty z predchádzajúceho kroku. Potom model spraví dopredný priechod a vypočíta sa strata (loss). Následne sa zavolá `loss.backward()`, čím sa pre všetky parametre vypočítajú gradienty. Nakoniec `optimizer.step()` upraví  (napr. váhy) modelu podľa týchto gradientov.

Je dôležité volať `optimizer.zero_grad()` pred každým novým krokom. V PyTorchi sa totiž gradienty štandardne pripočítavajú k už existujúcim hodnotám. Keby sa nevymazali, model by pri ďalšom kroku pracoval so starými aj novými gradientmi naraz, čo by viedlo k nesprávnej aktualizácii parametrov.

Táto štvorica príkazov

1. `optimizer.zero_grad()`
2. dopredný priechod modelom (`nn.Module`=>`forward`) a výpočet straty (`criterion`)
3. `loss.backward() `
4. `optimizer.step()`

tvorí jadro tréningovej slučky a opakuje sa pre každý batch počas učenia.

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cpu")
model = TinyClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

x = torch.randn(8, 2, device=device)
y = torch.randint(0, 2, (8,), device=device)

model.train()
optimizer.zero_grad()
logits = model(x)
loss = criterion(logits, y)
loss.backward()
optimizer.step()

print("loss po jednom kroku:", float(loss))


### Jeden krok učenia modelu v praxi

Nasledujúci kód ukazuje, ako sa v praxi vykoná jeden tréningový krok. Práve tu sa spoja pojmy, ktoré už boli vysvetlené skôr: model vytvorí výstup, funkcia straty zmeria chybu, spätné šírenie chyby vypočíta gradienty a optimalizátor podľa nich upraví parametre modelu.

**`device = torch.device("cpu")`** určuje, na akom zariadení sa bude výpočet vykonávať (`cpu`,`cuda:0`, `mps`, ...). V tejto ukážke sa používa `CPU`, aby fungovala rovnako na každom počítači.

**`model = TinyClassifier().to(device)`** vytvorí model a presunie jeho parametre na zvolené zariadenie. To znamená, že váhy aj bias budú uložené na `CPU`.

**`criterion = nn.CrossEntropyLoss()`** nastaví funkciu straty vhodnú pre klasifikáciu. Jej úlohou je porovnať výstup modelu so správnymi triedami a vypočítať, aká veľká je chyba.

**`optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)`** vytvorí optimalizátor Adam. Ten dostane všetky parametre modelu a bude ich počas učenia upravovať. Hodnota `lr` určuje, aký veľký krok sa pri tejto úprave spraví.

Potom sa vytvorí jedna malá ukážková dávka dát. **`torch.randn(8, 2, device=device)`** vytvorí 8 vstupov, pričom každý vstup má 2 číselné vlastnosti. **`torch.randint(0, 2, (8,), device=device)`** vytvorí 8 cieľových hodnôt, teda správnych tried `0` alebo `1`.

**`model.train()`** prepne model do tréningového režimu. Pri tomto jednoduchom modeli sa tým správanie výrazne nemení, ale pri zložitejších vrstvách, ako sú dropout alebo batch normalization, je to dôležité.

**`optimizer.zero_grad()`** vynuluje gradienty z predchádzajúceho kroku. V PyTorchi sa totiž gradienty automaticky pripočítavajú, preto ich treba pred novým krokom vymazať.

**`logits = model(x)`** vykoná dopredný prechod modelom. Vstupné dáta prejdú cez vrstvy modelu a vzniknú výstupné hodnoty pre jednotlivé triedy.

**`loss = criterion(logits, y)`** vypočíta stratu, teda jedno číslo, ktoré hovorí, aká veľká je chyba modelu na tejto dávke dát.

**`loss.backward()`** spustí **backpropagation**, teda spätné šírenie chyby. V tomto kroku sa chyba „šíri dozadu“ cez model a pre každý parameter sa vypočíta gradient. Inými slovami, model tým zistí, ktoré váhy prispeli k chybe a ako ich treba upraviť, aby sa chyba zmenšila.

**`optimizer.step()`** použije vypočítané gradienty a spraví jeden aktualizačný krok. Práve tu sa váhy modelu skutočne zmenia.

**`print("loss po jednom kroku:", float(loss))`** vypíše hodnotu straty v tomto kroku. Je to číselná informácia o tom, aká veľká bola chyba pred aktualizáciou parametrov.

Celý tento postup tvorí základ učenia neurónovej siete. Najprv sa vymažú staré gradienty, potom model vytvorí výstup, z neho sa vypočíta chyba, spätným šírením sa určia gradienty a napokon optimalizátor upraví parametre. Tento cyklus sa potom opakuje znova a znova pre ďalšie dávky dát.

### Ako tento krok vyzerá v kontexte epoch

V praxi sa jeden tréningový krok nevykoná len raz. Opakuje sa pre každý batch dát a celý tento proces sa potom opakuje ešte aj cez viacero epoch. Aby bolo jasné, čo tieto pojmy znamenajú, je dobré ich odlíšiť:

- **batch** je malá skupina vzoriek, ktoré model spracuje naraz. Jeho veľkosť sa nastavuje pri vytváraní `DataLoadera`, napríklad `DataLoader(ds, batch_size=100, shuffle=True)`. Práve parameter `batch_size` určuje, koľko vzoriek dostane model v jednej iterácii.
- **iterácia** je jeden tréningový krok nad jedným batchom
- **epócha** je jeden úplný prechod cez celý tréningový dataset

Ak teda máme napríklad **1000 obrázkov** a v `DataLoaderi` nastavíme **batch_size=100**, model ich počas jednej epóchy nespracuje naraz, ale rozdelí si ich na **10 batchov po 100 obrázkov**. To znamená:

- za **1 iteráciu** model spracuje **100 obrázkov**
- za **1 epóchu** model spracuje **všetkých 1000 obrázkov**
- počas **1 epóchy** prebehne **10 iterácií**

Všeobecne teda platí:

$$
\text{počet iterácií za jednu epóchu} = \frac{\text{počet vzoriek v datasete}}{\text{batch size}}
$$

Ak počet vzoriek nie je deliteľný veľkosťou batchu presne, posledný batch býva menší. Napríklad pri 1050 obrázkoch a `batch_size=100` bude mať jedna epócha 11 iterácií, pričom posledný batch bude obsahovať len 50 obrázkov.

Ak nastavíme napríklad **5 epoch**, znamená to, že model prejde celý dataset päťkrát. Pri datasete s 1000 obrázkami a `batch_size=100` to znamená:

- **10 iterácií na jednu epóchu**
- **5 epôch**
- spolu teda **50 iterácií**

Dá sa to predstaviť jednoducho aj slovne. Batch hovorí, **koľko obrázkov spracujem naraz**, a túto veľkosť určuje `DataLoader`. Iterácia je **jeden konkrétny krok učenia nad jedným batchom**. Epócha hovorí, **koľkokrát som už prešiel celý dataset od začiatku do konca**.

V kóde sa to prejaví ako dva vnorené cykly. Vonkajší cyklus ide cez epóchy a vnútorný cyklus ide cez jednotlivé batche z `DataLoadera`.

```python
import torch
import torch.nn as nn

device = torch.device("cpu")
model = TinyClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ukážkový DataLoader
# batch_size určuje, koľko vzoriek sa spracuje v jednej iterácii
loader = torch.utils.data.DataLoader(ds, batch_size=8, shuffle=True)

# počet úplných prechodov cez celý dataset
num_epochs = 5

model.train()

# vonkajší cyklus = epóchy
for epoch in range(num_epochs):
    epoch_loss = 0.0

    # vnútorný cyklus = jednotlivé batche z DataLoadera
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        # priebežné sčítanie straty za celú epóchu
        epoch_loss += loss.item()

    print(f"epocha {epoch + 1}, loss = {epoch_loss:.4f}")
```

## 6. Tréningová slučka, validácia a logovanie priebehu učenia

Doteraz boli jednotlivé časti vysvetlené osobitne: model vytvorí výstup, funkcia straty zmeria chybu, spätné šírenie chyby vypočíta gradienty a optimalizátor podľa nich upraví váhy. V praxi sa však tieto kroky nepoužívajú izolovane. Spájajú sa do opakujúcej sa tréningovej slučky, v ktorej model prechádza cez tréningové dáta po batchoch a počas viacerých epoch sa postupne učí lepšie nastavenie parametrov.

Popri samotnom tréningu sa zvyčajne sleduje aj správanie modelu na validačných dátach. To je dôležité preto, že nízka chyba na tréningovej množine sama osebe ešte neznamená, že model bude dobre fungovať aj na nových dátach. Tréningová a validačná fáza preto majú rozdielny účel. V tréningu model svoje parametre mení, vo validácii ich už nemení a iba meriame, ako dobre si poradí na dátach, ktoré počas učenia nepoužil na aktualizáciu váh.

S tým súvisia aj dva režimy modelu: `train()` a `eval()`. V režime `train()` sa model správa tak, ako treba počas učenia. To je dôležité najmä pri vrstvách ako dropout alebo batch normalization, ktoré sa počas tréningu a počas vyhodnocovania správajú odlišne. V režime `eval()` sa model prepne do stabilného vyhodnocovacieho režimu, aby sa výstupy pri validácii a neskoršom použití modelu správali konzistentne.

Pri validácii navyše nechceme počítať gradienty, pretože sa parametre modelu aj tak nemenia. Preto sa vyhodnocovanie často obalí do `torch.no_grad()`, čím sa zníži spotreba pamäte aj výpočtová réžia. Typický postup teda vyzerá tak, že v tréningu sa použije `model.train()`, vykoná sa dopredný prechod, výpočet straty, backpropagation a krok optimalizátora, zatiaľ čo pri validácii sa použije `model.eval()` a len sa vypočíta chyba bez aktualizácie parametrov.

Keď sa model učí počas viacerých epoch, je užitočné priebeh sledovať priebežne, nielen na konci. Práve na to slúži TensorBoard. Umožňuje zapisovať hodnoty, ako je tréningová strata alebo validačná strata, a potom ich zobraziť ako grafy v čase. Vďaka tomu viete vidieť, či model napreduje, či sa učenie zastavilo, alebo či sa začína objavovať preučenie, teda situácia, keď sa model zlepšuje na tréningových dátach, ale nie na validačných.

Počas učenia teda sledujeme tri veci naraz. Po prvé, model sa učí na tréningových batchoch a priebežne mení svoje parametre. Po druhé, na validačných dátach kontrolujeme, či sa zlepšuje aj na dátach, na ktorých sa priamo neučil. Po tretie, tieto hodnoty si zapisujeme, aby sme ich mohli neskôr zobraziť a porovnať. Až spolu dávajú tieto tri časti zmysluplný obraz o tom, čo sa počas učenia skutočne deje.

### Ako bude vyzerať kód v praxi

Nasledujúce dve kódové bunky ukazujú už bežnejší tvar tréningu modelu. V prvej sa pripravia dáta, rozdelenie na tréningovú a validačnú časť, `DataLoader`, model, funkcia straty a optimalizátor. Zároveň sa definujú dve pomocné funkcie. Jedna vykoná celý tréningový prechod cez všetky batchy v jednej epóche a druhá spočíta stratu na validačných dátach bez učenia.

V druhej kódovej bunke sa potom spustí samotná slučka cez epóchy. V každej epóche sa najprv model učí na tréningových dátach, potom sa vyhodnotí na validačných dátach a nakoniec sa obe hodnoty zapíšu do TensorBoardu. Takto už vzniká malá, ale úplná kostra tréningu, aká sa v PyTorchi používa aj pri reálnejších úlohách.

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

# Dataset a model (znovu definované, aby táto časť bola spustiteľná samostatne)
class Toy2DDataset(torch.utils.data.Dataset):
    def __init__(self, n=200, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.x = torch.randn(n, 2, generator=g) * 0.9 + 0.2
        self.y = (self.x[:, 1] > 0).long()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]


class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 2)

    def forward(self, x):
        return self.fc(x)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / max(n, 1)


@torch.no_grad()
def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / max(n, 1)


# Rozdelenie train / val a príprava modelu
full = Toy2DDataset(200, seed=1)
n_val = 40
train_ds, val_ds = random_split(
    full, [len(full) - n_val, n_val], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

device = torch.device("cpu")
model = TinyClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)



### Prvá časť kódu: dáta, model a pomocné funkcie pre tréning a validáciu

Táto časť pripraví všetko, čo je potrebné pred samotnou slučkou cez epóchy. Najprv sa importujú potrebné moduly, potom sa vytvorí dataset, rozdelí sa na tréningovú a validačnú časť, pripravia sa `DataLoader` objekty, model, funkcia straty a optimalizátor. Okrem toho sa definujú aj dve pomocné funkcie, ktoré oddeľujú tréning a vyhodnocovanie do prehľadných blokov.

Dataset `Toy2DDataset` a model `TinyClassifier` sú stále zámerne veľmi jednoduché. Cieľom tu nie je zložitosť architektúry, ale to, aby bolo dobre vidieť celú kostru učenia. Dataset vracia dvojice `(x, y)` a model z dvojrozmerného vstupu vytvorí dve výstupné hodnoty, teda jednu pre každú triedu.

Dôležitý krok je rozdelenie dát na tréningovú a validačnú časť pomocou `random_split`. Tréningová časť sa používa na učenie modelu, teda na aktualizáciu váh. Validačná časť slúži len na priebežnú kontrolu, ako sa model správa na dátach, na ktorých sa priamo neučil. Práve toto rozdelenie je dôležité, ak chceme odlíšiť skutočné zlepšovanie modelu od toho, že si len príliš dobre zapamätal tréningové dáta.

Z vytvorených podmnožín sa potom pripravia `DataLoader` objekty. Pri tréningových dátach je nastavené `shuffle=True`, aby model nevidel vzorky vždy v rovnakom poradí. Pri validačných dátach sa poradie miešať nemusí, preto je tam `shuffle=False`.

Funkcia `train_one_epoch` vykoná jeden celý tréningový prechod cez všetky batche tréningových dát. V každom batchi sa zopakuje rovnaký postup, ktorý už bol vysvetlený skôr: vynulovanie gradientov, dopredný prechod, výpočet straty, backpropagation a krok optimalizátora. Funkcia na konci nevracia stratu z jedného batchu, ale priemernú stratu za celú epóchu.

Funkcia `evaluate_loss` slúži na vyhodnotenie modelu na validačných dátach. Na rozdiel od tréningu sa tu model neučí, takže sa nepoužíva `optimizer.step()` ani `backward()`. Model sa len prepne do režimu `eval()`, spočíta sa strata na validačných batchoch a opäť sa vráti priemer za celú validačnú časť. Dekorátor `@torch.no_grad()` zabezpečí, že sa pri tomto výpočte nebudú počítať gradienty, čo je úspornejšie a zároveň to presne zodpovedá tomu, čo pri validácii chceme robiť.

Na konci tejto časti sú už pripravené všetky stavebné prvky potrebné na učenie. Máme tréningové a validačné dáta, model, funkciu straty, optimalizátor aj dve pomocné funkcie. V ďalšom kroku sa tieto časti spoja do samotnej slučky cez epóchy a výsledky sa budú priebežne zapisovať.

In [ ]:
log_dir = os.path.join("runs", "5_uvod_demo")
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir=log_dir)

epochs = 15
for epoch in range(1, epochs + 1):
    tr = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va = evaluate_loss(model, val_loader, criterion, device)
    writer.add_scalar("loss/train", tr, epoch)
    writer.add_scalar("loss/val", va, epoch)
    writer.add_scalar("meta/epoch_index", float(epoch), epoch)

writer.close()
print("Zapísané logy do:", log_dir)
print("Spustite: tensorboard --logdir runs")



### Druhá časť kódu: TensorBoard a slučka cez epóchy

V tejto časti sa už spustí samotné učenie modelu. Všetko, čo bolo pripravené predtým — dáta, `DataLoader`, model, funkcia straty, optimalizátor aj pomocné funkcie — sa teraz spojí do jednej opakujúcej sa slučky cez epóchy.

Objekt `SummaryWriter` slúži na zapisovanie priebehu učenia do TensorBoardu. Vďaka tomu sa dajú neskôr zobraziť grafy, napríklad ako sa v čase mení tréningová strata a validačná strata. To je praktické preto, že priebeh učenia nemusíme sledovať len cez výpis v konzole, ale vieme si ho aj vizuálne porovnať medzi jednotlivými epóchami.

Slučka `for epoch in range(...)` predstavuje opakovanie učenia cez viacero epoch. V každej epóche sa najprv zavolá funkcia `train_one_epoch`, ktorá prejde všetky tréningové batche a vráti priemernú tréningovú stratu za danú epóchu. Potom sa zavolá `evaluate_loss`, ktorá bez učenia spočíta priemernú validačnú stratu.

Tieto dve hodnoty sa následne zapíšu pomocou `writer.add_scalar(...)`. Jedna reprezentuje tréningovú chybu a druhá validačnú chybu. Keď sa takýto zápis opakuje v každej epóche, TensorBoard z nich vie vytvoriť grafy, na ktorých je vidieť, či sa model postupne zlepšuje.

Výpis pomocou `print(...)` slúži len ako priebežná textová kontrola počas behu. TensorBoard je vhodný na dlhodobejšie sledovanie priebehu, zatiaľ čo výpis do konzoly je užitočný na rýchlu orientáciu počas tréningu.

Na konci sa zavolá `writer.close()`, čím sa zapisovanie korektne ukončí. Tým je uzavretá celá základná kostra tréningu. Model sa učí na tréningových dátach, priebežne sa kontroluje na validačných dátach a výsledky sa zaznamenávajú tak, aby sa dali neskôr prezrieť aj graficky.

## 7. Ako o tejto kostre premýšľať pri zložitejších úlohách

Dôležité nie je zapamätať si konkrétny malý príklad, ale pochopiť jeho kostru. Tá sa pri zložitejších úlohách v zásade nemení. Stále máme dáta, spôsob ich načítania cez `Dataset` a `DataLoader`, model, funkciu straty, optimalizátor, tréningovú slučku a vyhodnocovanie na validačných dátach. To, čo sa mení, nie je samotný princíp, ale konkrétna podoba jednotlivých častí.

Pri jednoduchom príklade sú vstupy malé a model je zámerne veľmi jednoduchý. Pri reálnejších úlohách však môžu byť vstupom obrázky, text, zvuk alebo tabuľkové dáta s väčším počtom príznakov. Rovnako sa môže meniť aj cieľ úlohy. Niekedy ide o klasifikáciu, inokedy o regresiu, segmentáciu alebo inú predikčnú úlohu. Z toho potom vyplýva aj iný tvar vstupov, iný tvar výstupu modelu a niekedy aj iná funkcia straty.

Užitočné je preto sledovať, ktoré časti zostávajú rovnaké a ktoré sa prispôsobujú konkrétnemu problému. Rovnaký zostáva základný tok práce. Najprv sa načítajú dáta, potom model vytvorí výstup, funkcia straty porovná tento výstup so správnou odpoveďou, spätné šírenie chyby vypočíta gradienty a optimalizátor upraví parametre. Tento cyklus sa opakuje počas batchov a epoch bez ohľadu na to, či pracujeme s malým syntetickým príkladom alebo s reálnou úlohou.

To, čo sa prispôsobuje, je najmä definícia jednej vzorky v `Dataset`, spôsob batchovania v `DataLoaderi`, architektúra modelu, počet vstupných a výstupných rozmerov, typ funkcie straty a spôsob vyhodnotenia výsledkov. Pri klasifikácii nás zaujíma napríklad presnosť, pri regresii chyba predikcie, pri segmentácii zas zhoda predikovanej masky so správnou maskou.

Je preto dobré čítať ďalšie príklady nie ako úplne nové svety, ale ako obmeny tej istej kostry. Keď sa v novom príklade objaví iný dataset alebo zložitejší model, netreba sa stratiť v detailoch. Najprv má zmysel položiť si niekoľko základných otázok. Čo je jedna vzorka. Aký tvar má vstup. Čo má model predpovedať. Aký tvar má výstup. Aká strata porovnáva výstup so správnou odpoveďou. A kde sa v kóde vykonáva tréningový krok.

Ak si tieto body viete v novom príklade nájsť, už sa v ňom viete orientovať. Práve v tom spočíva hlavný význam tejto úvodnej kostry. Nie je dôležitá sama osebe, ale ako opakovateľný vzor, podľa ktorého sa dajú čítať a vytvárať aj podstatne zložitejšie riešenia.

## 8. Šablóna v tomto notebooku verzus 5a a 5b: čo zmeniť a prečo

Tento notebook (`5_uvod_do_pytorch.ipynb`) ukazuje **minimum**: malý `Dataset`, jednoduchý model, jeden optimalizačný krok, potom kratší ale úplný príklad s epochami a TensorBoardom. Súbory **5a** a **5b** tú istú logiku **rozšíria** o reálne CSV, iné rozmery a inú úlohu (klasifikácia vs. regresia). Nižšie je to rozdelené tak, aby ste vedeli, **ktorú časť kopírovať** a **čo nahradiť**.

### Tri vrstvy

**A) Šablóna v `5_uvod_do_pytorch.ipynb`:** učenie sa z náhodných alebo hračkových dát, jednoduchá architektúra, dôraz na pochopenie príkazov `zero_grad`, `backward`, `step`, `train`/`eval`, `DataLoader`, TensorBoard.

**B) `5a.ipynb`:** rovnaká kostra, ale vstupy sú **tri súradnice z CSV** a výstup je **počet tried** z tabuľky; strata ostáva **`CrossEntropyLoss`**; rozdelenie dát je **train / val / test** so stratifikáciou podľa tried; pribudnú **presnosť**, **zmätková matica** a **3D graf**.

**C) `5b.ipynb`:** vstup je **jedna** súradnica `x`, výstup siete je **jedno číslo** (predikcia `y`); strata je **`MSELoss`**; indexy train/test sú v **samostatnom súbore** `datafunindx.csv`; často sa pracuje s **celými tenzormi** naraz a s voľbou **Adam vs LBFGS** (LBFGS používa **closure**).

### Tabuľka: čo sa líši (a prečo)

| Oblast | V šablóne `5_uvod_do_pytorch.ipynb` | V `5a.ipynb` (klasifikácia) | V `5b.ipynb` (regresia) |
|--------|---------------------|------------------------------|-------------------------|
| Vstup do siete | 2 čísla (2D bod) | 3 čísla (`x,y,z` z CSV) | 1 číslo (`x`) |
| Výstup poslednej vrstvy | `num_classes` (napr. 2 logity) | `num_classes` podľa značiek v dátach | **1** neurón (predikcia reálneho čísla) |
| Strata | `CrossEntropyLoss` | `CrossEntropyLoss` | **`MSELoss`** (stredná štvorcová chyba medzi predikciou a `y`) |
| Označenia `y` | celé čísla tried | celé indexy tried | **float** cieľová hodnota (po normalizácii) |
| Dáta do modelu | `DataLoader`, batchy | `DataLoader` + `Subset` podľa indexov | Často **tensors** `Xt`, `Yt` na `device` (celý tréning naraz) |
| Rozdelenie dát | `random_split` | Fixný test, potom val zvyšku, stratifikácia | Podľa **stĺpca v `datafunindx.csv`** (train vs test) |
| Validácia | áno (val loader) | áno (train / val / test) | V základnom skripte hlavne **train vs test** (iná štruktúra zadania) |
| Optimalizátor | Adam po batchoch | Adam po batchoch | **Adam alebo LBFGS**; pri LBFGS **`optimizer.step(closure)`** |

Posledný riadok je kľúčový: **LBFGS** v PyTorch očakáva funkciu **closure**, ktorá znovu vypočíta stratu a gradient v rámci jedného kroku — preto slučka v 5b **nie je** identická s Adamom po batchoch, hoci **myšlienka** (minimalizovať stratu) je tá istá.

### Kód: čo v hlavičke úlohy vymeníte najčastejšie

**Výstupná vrstva** — z viactriednej klasifikácie do regresie:

```python
# klasifikácia (5a, podobne ako v šablóne)
self.out = nn.Linear(hidden_dim, num_classes)  # num_classes >= 2
criterion = nn.CrossEntropyLoss()
# y musí byť torch.long s tvarom (batch,) s hodnotami 0..num_classes-1

# regresia (5b)
self.out = nn.Linear(hidden_dim, 1)
criterion = nn.MSELoss()
# y musí mať rovnaký tvar ako predikcia, typicky float, tvar (batch, 1) alebo (batch,)
```

**Slučka s Adamom** (spoločná pre šablónu a 5a; zjednodušený tvar ako v 5b pri voľbe Adam):

```python
optimizer.zero_grad()
pred = model(x)
loss = criterion(pred, y)
loss.backward()
optimizer.step()
```

**LBFGS v 5b** (zjednodušená podstata — v skutočnom súbore je `closure` vnútri slučky po epochách):

```python
def closure():
    optimizer.zero_grad()
    pred = model(Xt)
    loss = criterion(pred, Yt)
    loss.backward()
    return loss

optimizer.step(closure)
```

### Recept: zo šablóny tohto notebooku k úlohe 5a

Načítajte CSV namiesto náhodných bodov: v `Dataset.__getitem__` vráťte riadok z matice príznakov a zodpovedajúci index triedy. Nastavte `input_dim=3` a `output_dim` podľa počtu tried. Rozdeľte indexy tak, aby test bol fixný a triedy boli zastúpené v train aj val. Zachovajte **`train_one_epoch`** a **`evaluate`** s `CrossEntropyLoss`. TensorBoard nech loguje `loss` a **`accuracy`** ak ju počítate (v 5a sa presnosť počíta v tých istých funkciách).

### Recept: zo šablóny k úlohe 5b

Zmeňte poslednú vrstvu na **jeden výstup**, `criterion` na **`MSELoss`**, cieľové `y` na float. Načítajte `x` a `y` z `datafun.csv` a indexy z `datafunindx.csv`. Ak chcete **LBFGS**, nahraďte batch slučku prácou s maticami `Xt`, `Yt` a použite **closure**; ak **Adam**, môžete ponechať podobnú slučku ako v šablóne, len s tvarom tenzorov zodpovedajúcim regresii. Normalizujte `x` aj `y` podľa štatistík **len z tréningu**, potom rovnako škálujte test.

### Zhrnutie

**Nemenné** medzi všetkými troma súbormi: zmysel **`nn.Module`**, existencia **straty** a **optimalizátora**, potreba **`zero_grad` / backward / step`** pri učení, oddelenie **tréningu** od **vyhodnotenia** (`eval`, `no_grad`), zápis priebehu cez **TensorBoard** ak chcete vidieť krivky.

**Menné** sú presne tie časti, ktoré sú v tabuľke: rozmer vstupu a výstupu, typ straty, typ `y`, spôsob načítania a rozdelenia dát, a tvar slučky pri LBFGS oproti Adamovi po batchoch.


## 8. Čo sa pri prechode na konkrétnejšiu úlohu mení a čo zostáva rovnaké

Pri prechode od jednoduchého ukážkového príkladu ku konkrétnejšej úlohe sa nemení základný princíp práce, ale mení sa podoba niektorých jeho častí. Stále platí rovnaká kostra: `Dataset` určuje, čo je jedna vzorka, `DataLoader` skladá vzorky do batchov, model vytvorí predikciu, funkcia straty porovná predikciu so správnou odpoveďou, backpropagation vypočíta gradienty a optimalizátor upraví parametre modelu.

To, čo sa prispôsobuje, je najmä typ vstupu, význam cieľa `y`, architektúra modelu, posledná vrstva a funkcia straty. Inak vyzerá klasifikácia a inak regresia, ale základná kostra učenia zostáva rovnaká.

### Klasifikácia verzus regresia

Najvýraznejší rozdiel býva v tom, **čo má model predpovedať**.

Pri klasifikácii model rozhoduje medzi triedami. Na výstupe preto potrebuje jednu hodnotu pre každú triedu a cieľ `y` predstavuje index správnej triedy.

Pri regresii model nevyberá triedu, ale predpovedá číselnú hodnotu. Výstupná vrstva má preto typicky jeden neurón a cieľ `y` musí mať rovnaký tvar ako predikcia.

```python
# klasifikácia (5a, podobne ako v šablóne)
self.out = nn.Linear(hidden_dim, num_classes)  # num_classes >= 2
criterion = nn.CrossEntropyLoss()
# y musí byť torch.long s tvarom (batch,) s hodnotami 0..num_classes-1

# regresia (5b)
self.out = nn.Linear(hidden_dim, 1)
criterion = nn.MSELoss()
# y musí mať rovnaký tvar ako predikcia, typicky float, tvar (batch, 1) alebo (batch,)
```

Na tomto krátkom porovnaní vidno, že sa nemení celý spôsob učenia, ale najmä:

* posledná vrstva modelu
* funkcia straty
* očakávaný tvar a typ cieľových hodnôt `y`

### Čo zostáva rovnaké v tréningovom kroku

Pri bežnom trénovaní s optimalizátorom ako `Adam` zostáva samotný tréningový krok veľmi podobný bez ohľadu na to, či ide o klasifikáciu alebo regresiu. Najprv sa vynulujú gradienty, potom model vytvorí predikciu, vypočíta sa strata, vykoná sa backpropagation a nakoniec optimalizátor upraví parametre.

```python
optimizer.zero_grad()
pred = model(x)
loss = criterion(pred, y)
loss.backward()
optimizer.step()
```

Toto je základná kostra učenia, ktorá sa opakuje stále dookola v jednotlivých batchoch a epóchach. Keď sa v inom príklade zmení model alebo strata, táto časť často zostane takmer rovnaká.


### Kedy sa tréningový krok mení viac

Pri väčšine bežných príkladov sa používa optimalizátor ako `SGD` alebo `Adam`, kde má tréningový krok veľmi jednoduchý tvar. Niekedy sa však nemení len model alebo funkcia straty, ale aj samotný spôsob optimalizácie.

V kontexte úlohy 5b nejde o klasifikáciu, ale o **regresiu**. Cieľom je, aby sa neurónová sieť naučila správanie nejakého systému na základe nameraných dvojíc **vstup – výstup**. Inými slovami, sieť sa snaží vytvoriť aproximáciu daného systému tak, aby po zadaní vstupu vedela čo najpresnejšie odhadnúť jeho výstup. Cieľom teda nie je len prispôsobiť sa tréningovým dátam, ale naučiť sieť tak, aby sa po natrénovaní správala podobne ako pôvodný systém. Dá sa na to pozerať ako na jednoduchú formu **identifikácie systému**, kde sa neurónová sieť používa ako jeho dátovo naučený model alebo simulácia.

Práve pri takomto type úloh môže dávať zmysel aj `LBFGS`, najmä ak je problém menší a cieľom je čo najpresnejšie doladiť parametre modelu. Na rozdiel od `Adam` alebo `SGD` sa `LBFGS` nespráva tak, že v každom kroku len raz pozrie na gradient a hneď spraví update. Počas jedného kroku si môže potrebovať stratu a gradient prepočítať aj viackrát, aby lepšie odhadol vhodný smer a veľkosť kroku.

Preto `LBFGS` nepoužíva úplne rovnaký zápis tréningového kroku ako `Adam`. Namiesto priameho tvaru

```python
optimizer.zero_grad()
pred = model(x)
loss = criterion(pred, y)
loss.backward()
optimizer.step()
````

sa pri `LBFGS` výpočet straty a gradientov uzavrie do pomocnej funkcie `closure`. Táto funkcia vie znovu vypočítať dopredný priechod, stratu aj gradienty vždy, keď to optimalizátor počas jedného kroku potrebuje.

```python id="67uz3n"
def closure():
    optimizer.zero_grad()
    pred = model(Xt)
    loss = criterion(pred, Yt)
    loss.backward()
    return loss

optimizer.step(closure)
```

Zmysel tejto konštrukcie je praktický. Optimalizátor `LBFGS` si počas jedného kroku môže viackrát vyžiadať aktuálnu hodnotu straty a gradientov. Preto ich nemáme zapísané priamo vo voľnom kóde ako pri `Adam`, ale zabalené do funkcie, ktorú vie optimalizátor opakovane zavolať.

Dá sa to chápať aj takto. Pri `Adam` je jeden krok pomerne priamočiary. Pozrieme sa na gradient, spravíme update a ideme ďalej. Pri `LBFGS` je jeden krok premyslenejší a výpočtovo náročnejší. Optimalizátor sa snaží z dostupných informácií lepšie odhadnúť, aký krok bude výhodný, a preto si môže v rámci jedného update vyžiadať viacnásobné prepočítanie straty a gradientov.

Najdôležitejšie je všimnúť si, že ani tu sa nemení základná logika učenia. Model stále vytvorí predikciu, strata ju porovná so správnou odpoveďou, backpropagation vypočíta gradienty a optimalizátor upraví parametre. Mení sa len to, **ako presne je tento krok zapísaný v kóde** a **koľkokrát si optimalizátor počas jedného kroku potrebuje prepočítať stratu a gradienty**.
